In [62]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber
import extract_exercises_from_exam_file as parse
from random import choice

In [63]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [64]:
def load_keywords(json_file=ROOT / "json/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [85]:
def classify_exercise(statement: str, keywords = None) -> list:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    topics = []
    for topic in keywords.keys():
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        if re.search(pattern, statement):
            topics.append(topic)

    return topics

In [86]:
def create_df_exercises_from_exam(exam_file: Path) -> pd.DataFrame:
    exercises_list = []

    # extraer el texto del examen
    exercises_text = parse.extract_exercises_from_pdf(exam_file)

    # clasificar el periodo del examen
    year = int(re.search(r"\d{4}", exam_file.name)[0])
    exam_period = parse.get_exam_period(year)

    # anadir los ejercicios a una lista de diccionarios para
    # luego crear el dataframe
    for ex in parse.split_exercises(exercises_text, exam_period):
        topic = classify_exercise(statement=ex)
        exercises_list.append({"year" : year, "exam" : exam_file.name, "topics" : topic, "statement" : ex})
              
    return pd.DataFrame(exercises_list)

Clasificar ejercicios de un archivo

In [67]:
file = ROOT / "examenes_quimica/2025_quimica/2025_quimica_Quimica - Examen_Titular-A_Quimica.pdf"

In [68]:
create_df_exercises_from_exam(file).explode("topics")

,year,exam,topics,statement
0,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,configuracion_electronica,1A. Escriba las configuraciones electrónicas d...
0,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,propiedades_periodicas,1A. Escriba las configuraciones electrónicas d...
1,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,enlace_quimico,"1B. a) Dadas las moléculas H₂S y PF₃, razone e..."
2,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,termoquimica,2A. Justifique si son verdaderas o falsas las ...
3,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,cinetica,2B. La reacción química 2A + B → C tiene como ...
4,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,equilibrio,3A. El equilibrio de descomposición del NaHCO₃...
5,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,acido-base,3B. Se preparan 250 mL de una disolución acuos...
6,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,formulacion,4A. Nombre o formule los siguientes compuestos...
7,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,NaN,4B. a) Escriba y ajuste la reacción de combust...
8,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,solubilidad,PREGUNTA 5.-\nPROTECCIÓN CONTRA LA CORROSIÓN\n...


In [106]:
exam_files = list((ROOT / "examenes_quimica").rglob("*.pdf"))


In [107]:
for _ in range(20):
    random_exam = choice(exam_files)
    exam_files.remove(random_exam)
    df = create_df_exercises_from_exam(random_exam).explode("topics")
    if df.topics.isna().sum() > 0:
        print(random_exam.name)
        print(df[df["topics"].isna()].statement.values[0])

química - 2014 - reserva 4.pdf
2, opción A Justifique la veracidad o falsedad de las siguientes afirmaciones, referidas a la siguiente
ecuación: 2SO₂ (g) + O₂ (g) → 2SO₃ (g).
a) Dos moles de SO₂ reaccionan con una molécula de oxígeno, para dar dos moléculas de SO₃ .
b) En las mismas condiciones de presión y temperatura, dos litros de SO₂ reaccionan con un litro
de O₂ para dar dos litros de SO₃ .
c) Cuatro moles de SO₂ reaccionan con dos moles de O₂ para dar cuatro moles de SO₃ .


química - 2016 - reserva 3.pdf
2, opción A En un matraz cerrado de 5 L hay 42 g de N₂ a 27ºC.
a) Determine la presión en el interior del matraz.
b) Se deja salir nitrógeno hasta que la presión interior sea de 1 atm. Calcule cuántos gramos de N₂ han
salido del matraz.
c) ¿A qué temperatura deberíamos poner el recipiente para recuperar la presión inicial?
Dato: Masa atómica N=14. R = 0,082 atm·L·moI⁻¹·K⁻¹.




In [118]:
df_examenes_oficiales_clasificados = pd.DataFrame()
for file in exam_files:
    df = create_df_exercises_from_exam(random_exam).explode("topics")
    df_examenes_oficiales_clasificados = pd.concat([df_examenes_oficiales_clasificados, df], axis=0)

In [119]:
df_examenes_oficiales_clasificados.to_csv("examenes_oficiales_clasificados.csv", index = False)

In [120]:
df_examenes_oficiales_clasificados[df_examenes_oficiales_clasificados.topics.isna()].to_csv("ejercicios_sin_clasificar.csv", index=False)

In [121]:
df_examenes_oficiales_clasificados[df_examenes_oficiales_clasificados.topics.isna()]

,year,exam,topics,statement
